# Nexus Packages Installation

## Option 1: Run Commands
```
uv add nexusformat pynxtools
uv sync
```
## Option 2: Update pyproject.toml
Add the following lines to the `pyproject.toml`
```toml
example-nxs = [
    "nexusformat>=2.0.2",
    "pynxtools>=0.13.2",
]
```
And then run `uv sync --all-extras`

## Creating .nxs with nexusformat & pynxtools
This example creates a mock .nxs file.
I would recommend using the tutorials at https://fairmat-nfdi.github.io/pynxtools/getting-started.html if you want use the nexus format more generally.

In [ ]:
from pathlib import Path

In [2]:
###
# Enter setup information
###
folder_path=Path("../project_a") # folder with images
#
file_ending=".jpg" # file extension
file_contains="" # file names must contain - i.e. "sample1-5x", "sample2", "10x", etc.
file_notcontains="" # file names must not contain - i.e. "calibration"
###
file_paths=[x for x in folder_path.iterdir() if x.suffix==file_ending]
print(file_paths)

[WindowsPath('../project_a/image_a01.jpg'), WindowsPath('../project_a/image_a02.jpg'), WindowsPath('../project_a/image_a03.jpg'), WindowsPath('../project_a/image_a04.jpg'), WindowsPath('../project_a/image_a05.jpg'), WindowsPath('../project_a/image_a06.jpg'), WindowsPath('../project_a/image_a07.jpg'), WindowsPath('../project_a/image_a08.jpg'), WindowsPath('../project_a/image_a09.jpg'), WindowsPath('../project_a/image_a10.jpg')]


In [ ]:
from pynxtools.dataconverter.readers.utils import parse_yml
import numpy as np
from PIL import Image
from nexusformat.nexus import NXfield, NXdata, NXgroup
CONVERT_DICT={
    "unit": "@units",
    "version": "@version",
    # "user": "USER[user]",
    "instrument": "INSTRUMENT[instrument]",
    # "detector": "DETECTOR[detector]",
    # "sample": "SAMPLE[sample]",
}
class SimpleReader():
    def handle_rgb_image_files(self, image_paths: list[Path]):
        self.images_data=dict()
        for image_path in image_paths:
            with Image.open(image_path) as PIL_image:
                im = np.array(PIL_image.convert('RGB'))
            intensity = NXfield(im, name="intensity")
            axis_i = NXfield(range(intensity.shape[0]),name='axis_i')
            axis_j = NXfield(range(intensity.shape[1]),name='axis_j')
            axis_k = NXfield(range(intensity.shape[2]),name='axis_k')
            data=NXdata(
                signal=intensity,
                axes=(axis_i, axis_j),
                axis_k = axis_k
            )
            self.images_data[image_path.name]=NXgroup(image_3d=data,name=f"image_{image_path.name}",nxclass="NXimage")
        return {}
    def handle_eln_file(self, file_path):
        self.eln_data = parse_yml(
            file_path,
            convert_dict=CONVERT_DICT,
            parent_key="/ENTRY[entry]",
        )
        return {}

reader=SimpleReader()
reader.handle_rgb_image_files(file_paths)
# for key, value in reader.images_data.items():  
#     print(f"  {key}  =  {value.tree}","\n---")


yml_path=Path("project_a1.yaml")
reader.handle_eln_file(yml_path)
# for key, value in reader.eln_data.items():  
#     print(f"  {key}  =  {value}")

# https://fairmat-nfdi.github.io/pynxtools/tutorial/writing-an-application-definition.html#the-experiment
# nyaml2nxdl example/project_a1_nexus/NXoptical_microscope.yaml --output-file example/project_a1_nexus/NXoptical_microscope.nxdl.xml
# Move copy to .venv\Lib\site-packages\pynxtools\definitions\contributed_definitions
# Good with that so: dataconverter generate-template --nxdl NXoptical_microscope

In [ ]:
from hdf5plugin import version as hdf5_version
from nexusformat import __version__ as nexus_version
from h5py.version import version as h5py_version

In [ ]:
from nexusformat.nexus import NXroot, NXentry

In [ ]:
r = NXroot()
for key,value in {
    "/@HDF5_Version[@hdf5_version]": hdf5_version,
    "/@HDF_version[@hdf_version]": hdf5_version,
    "/@NeXus_release[@nexus_release]": nexus_version,
    "/@NeXus_repository[@nexus_repository]": "nexusformat",
    "/@NeXus_version[@nexus_version]": nexus_version,
    "/@XML_version[@xml_version]": '1.0',
    "/@creator": "",
    "/@creator_version": "",
    "/@default": "",
    "/@file_name": "",
    "/@file_time": "",
    "/@file_update_time": "",
    "/@h5py_version": h5py_version,
    "/@partial": "",
    }.items():
    r.attrs[key]=NXfield(value=value)
entry=r["ENTRY[entry]"]=NXentry()
instrument=entry["INSTRUMENT[instrument]"]=NXgroup(name='instrument', nxclass='NXinstrument')
instrument["microscope"]=NXgroup(name='microscope', nxclass='NXcomponent')
instrument["detector"]=NXgroup(name='detector', nxclass='NXdetector')
instrument["objective"]=NXgroup(name='objective', nxclass='NXoptical_lens')
for key,value in reader.eln_data.items():
    r[key]=value
images=entry["images"]=NXgroup(name='images',nxclass='NXgroup')
for key, value in reader.images_data.items():  
    # print(f"  {key}  =  {value.tree}","\n---")
    images[f"image{key}[imageid]"]=value
print(r.tree)

root:NXroot
  @/@HDF5_Version[@hdf5_version] = '6.0.0'
  @/@HDF_version[@hdf_version] = '6.0.0'
  @/@NeXus_release[@nexus_release] = '2.0.2'
  @/@NeXus_repository[@nexus_repository] = 'nexusformat'
  @/@NeXus_version[@nexus_version] = '2.0.2'
  @/@XML_version[@xml_version] = '1.0'
  @/@creator = ''
  @/@creator_version = ''
  @/@default = ''
  @/@file_name = ''
  @/@file_time = ''
  @/@file_update_time = ''
  @/@h5py_version = '3.16.0'
  @/@partial = ''
  ENTRY[entry]:NXentry
    INSTRUMENT[instrument]:NXinstrument
      detector:NXdetector
        description = 'PAXcam2 LM camera'
      microscope:NXcomponent
        description = 'Olympus GX41 Compact Inverted Metallurgical Microscope'
      objective:NXoptical_lens
        description = '5x BF Mplan Achro 0.15NA'
        magnification = 5
        numerical_aperture = 0.15
    definition = 'NXoptical_microscopy'
    images:NXgroup
      imageimage_a01.jpg[imageid]:NXimage
        image_3d:NXdata
          @axes = ['axis_i', 'axis_j']

In [ ]:
r.save("project_a.nxs")

NXroot('project_a_nexus')

# Discover From `.nxs` and create project

In [55]:
import h5py

In [64]:
visit_list=list()
with h5py.File("project_a.nxs") as f:
    f.visit(visit_list.append)
# print(visit_list)
path_list=[path for path in visit_list if "image_3d/intensity" in path]
# print (path_list)
name_index=2
start_position=5
end_position=-9
path_dict={path.split("/")[name_index][start_position:end_position]:path for path in path_list}
print (path_dict)

{'image_a01.jpg': 'ENTRY[entry]/images/imageimage_a01.jpg[imageid]/image_3d/intensity', 'image_a02.jpg': 'ENTRY[entry]/images/imageimage_a02.jpg[imageid]/image_3d/intensity', 'image_a03.jpg': 'ENTRY[entry]/images/imageimage_a03.jpg[imageid]/image_3d/intensity', 'image_a04.jpg': 'ENTRY[entry]/images/imageimage_a04.jpg[imageid]/image_3d/intensity', 'image_a05.jpg': 'ENTRY[entry]/images/imageimage_a05.jpg[imageid]/image_3d/intensity', 'image_a06.jpg': 'ENTRY[entry]/images/imageimage_a06.jpg[imageid]/image_3d/intensity', 'image_a07.jpg': 'ENTRY[entry]/images/imageimage_a07.jpg[imageid]/image_3d/intensity', 'image_a08.jpg': 'ENTRY[entry]/images/imageimage_a08.jpg[imageid]/image_3d/intensity', 'image_a09.jpg': 'ENTRY[entry]/images/imageimage_a09.jpg[imageid]/image_3d/intensity', 'image_a10.jpg': 'ENTRY[entry]/images/imageimage_a10.jpg[imageid]/image_3d/intensity'}


In [65]:
from misalign.model.project import MISProjectJSON
from misalign.model.hdf5 import MISImageHDF5

In [66]:
mis_project=MISProjectJSON(
    images=[MISImageHDF5(
        hdf5_filepath="project_a.nxs",
        image_name=name,
        hdf5path=path,
        PIL_mode="RGB",)
        for name,path in path_dict.items()]
    )
print(mis_project)

A misalign project with:
Images:
    image_a01.jpg
    image_a02.jpg
    image_a03.jpg
    image_a04.jpg
    image_a05.jpg
    image_a06.jpg
    image_a07.jpg
    image_a08.jpg
    image_a09.jpg
    image_a10.jpg
Relations:

Calibration:

Project Path:
    None


In [ ]:
mis_project.save("project_a_nexus.mis.json")

### Add Relations

In [ ]:
from misalign.model.project import MISProjectJSON

In [70]:
mis_project_example=MISProjectJSON.load("../project_a/project_a-relations-calibrated.mis.json")
mis_project=MISProjectJSON.load("project_a_nexus.mis.json")

In [71]:
mis_project.set_relations(mis_project_example.get_relations())
mis_project.set_calibration(mis_project_example.get_calibration())
mis_project.save("project_a_nexus-rel-cal.mis.json")